In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "sarvamai/sarvam-1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Set the pad token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Base model loaded")

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Base model loaded


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

=========================<br>
🚀 STAGE 1 — LANGUAGE ADAPTATION<br>
=========================

In [ ]:
import json

pairs = []

path = "/content/drive/MyDrive/kumaoni_dataset_final.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        pairs.append((row["english"], row["kumaoni"]))

print(len(pairs))

9538


In [ ]:
stage1_data = []

for eng, kumaoni in pairs:

    text = f"""Translate English to Kumaoni.

English: {eng}

Kumaoni: {kumaoni}
"""

    stage1_data.append(text)

In [ ]:
from datasets import Dataset

dataset_stage1 = Dataset.from_dict({"text": stage1_data})

In [ ]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding=False
    )

    # tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset_stage1 = dataset_stage1.map(tokenize, remove_columns=["text"], load_from_cache_file=False)

Map:   0%|          | 0/9538 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args_stage1 = TrainingArguments(

    output_dir="/content/drive/MyDrive/kumaoni_stage1",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    num_train_epochs=2,

    learning_rate=2e-4,

    logging_steps=50,

    save_strategy="epoch",

    fp16=True,

    optim="paged_adamw_8bit",

    report_to="none"
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer_stage1 = Trainer(
    model=model,
    args=training_args_stage1,
    train_dataset=dataset_stage1,
    data_collator=data_collator
)

trainer_stage1.train()

Step,Training Loss
50,2.372208
100,1.795351
150,1.718442
200,1.650363
250,1.537796
300,1.538975
350,1.539128
400,1.457662
450,1.480167
500,1.466740


TrainOutput(global_step=2386, training_loss=1.3847520769092083, metrics={'train_runtime': 2182.491, 'train_samples_per_second': 8.74, 'train_steps_per_second': 1.093, 'total_flos': 1.8544100533272576e+16, 'train_loss': 1.3847520769092083, 'epoch': 2.0})

=========================<br>
🚀 STAGE 2 — CHAT ALIGNMENT<br>
=========================

In [ ]:
chat_data = []

chat_path = "/content/drive/MyDrive/kumaoni_chat_dataset_10k.jsonl"

with open(chat_path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)

        text = f"""### Instruction:
Respond in Kumaoni language.

### Input:
{row["input"]}

### Response:
{row["output"]}
"""

        chat_data.append(text)

len(chat_data)

10000

In [ ]:
dataset_stage2 = Dataset.from_dict({"text": chat_data})

In [ ]:
dataset_stage2 = dataset_stage2.map(tokenize, remove_columns=["text"])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
training_args_stage2 = TrainingArguments(

    output_dir="/content/drive/MyDrive/kumaoni_stage2",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    num_train_epochs=3,

    learning_rate=1e-4,

    logging_steps=50,

    save_strategy="epoch",

    fp16=True,

    optim="paged_adamw_8bit",

    report_to="none"
)

In [ ]:
trainer_stage2 = Trainer(
    model=model,
    args=training_args_stage2,
    train_dataset=dataset_stage2,
    data_collator=data_collator
)

trainer_stage2.train()

Step,Training Loss
50,0.769581
100,0.112519
150,0.079641
200,0.078011
250,0.077499
300,0.071681
350,0.071671
400,0.072469
450,0.071137
500,0.070250


TrainOutput(global_step=3750, training_loss=0.07858806660970052, metrics={'train_runtime': 3065.4018, 'train_samples_per_second': 9.787, 'train_steps_per_second': 1.223, 'total_flos': 2.2310580905312256e+16, 'train_loss': 0.07858806660970052, 'epoch': 3.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/kumaoni_chat_model")
tokenizer.save_pretrained("/content/drive/MyDrive/kumaoni_chat_model")

('/content/drive/MyDrive/kumaoni_chat_model/tokenizer_config.json',
 '/content/drive/MyDrive/kumaoni_chat_model/chat_template.jinja',
 '/content/drive/MyDrive/kumaoni_chat_model/tokenizer.json')

In [ ]:
!zip -r kumaoni_chat_model.zip /content/drive/MyDrive/kumaoni_chat_model

  adding: content/drive/MyDrive/kumaoni_chat_model/ (stored 0%)
  adding: content/drive/MyDrive/kumaoni_chat_model/README.md (deflated 66%)
  adding: content/drive/MyDrive/kumaoni_chat_model/adapter_model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/kumaoni_chat_model/adapter_config.json (deflated 58%)
  adding: content/drive/MyDrive/kumaoni_chat_model/chat_template.jinja (deflated 62%)
  adding: content/drive/MyDrive/kumaoni_chat_model/tokenizer_config.json (deflated 47%)
  adding: content/drive/MyDrive/kumaoni_chat_model/tokenizer.json (deflated 84%)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "sarvamai/sarvam-1"

adapter_path = "/content/drive/MyDrive/kumaoni_chat_model"  # path where you saved after stage 2

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)

model.eval()

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/193 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
def chat(user_input):

    prompt = f"""You are a helpful Kumaoni assistant.
### Instruction:
Respond only in Kumaoni language

### Input:
{user_input}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.45,
        top_p=0.8,
        do_sample=True,
        repetition_penalty=1.25,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("### Response:")[-1].split("###")[0].strip()

    print("Bot:", answer)

In [ ]:
chat("how are you?")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mi theek chhu, tum kas ho!


In [ ]:
chat("where is your house?")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mero ghar gaun ma chhu.


In [ ]:
chat("what are you doing?")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mi kaam ma lag chhu.


In [ ]:
chat("how is farming now?")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: farming tho theek chha!


In [ ]:
chat("hello?")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: namaskar!


In [ ]:
chat("who are you")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mi ek helpin kaam ma chhuuo


In [ ]:
chat("where do you live")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mi gaun ma ra chhu


In [ ]:
chat("what is AI")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: AI, social media ma kaam le thodo help lag chha.


In [ ]:
chat("who is modi")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: modal social ka human chha


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "sarvamai/sarvam-1"

adapter_path = "/content/drive/MyDrive/kumaoni_chat_model"  # your trained adapter

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)

model.eval()

print("Model loaded")

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Model loaded


In [ ]:
model = model.merge_and_unload()

model.save_pretrained("/content/drive/MyDrive/kumaoni_full_model")
tokenizer.save_pretrained("/content/drive/MyDrive/kumaoni_full_model")

print("Merged model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved


In [ ]:
model_path = "/content/drive/MyDrive/kumaoni_full_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Merged model loaded")

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Merged model loaded


In [ ]:
system_prompt = """
You are KumaoniGPT.

You are a friendly assistant that speaks Kumaoni language.
Always reply in Kumaoni.
Your answers should be short, natural and conversational.
You are a helpful assistant.
"""

In [ ]:
conversation_history = system_prompt

In [ ]:
def chat(user_input):

    global conversation_history

    prompt = f"""{conversation_history}

User: {user_input}
Assistant:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.45,
        top_p=0.8,
        repetition_penalty=1.25,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Assistant:")[-1].strip()

    conversation_history += f"\nUser: {user_input}\nAssistant: {answer}"

    print("Bot:", answer)

In [ ]:
chat("hello")
chat("where is your house?")
chat("what are you doing?")
chat("how is farming now?")
chat("who are you")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: namaskar!


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mero ghar gaun ma chhu.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mi kaam ma lag chhu.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: farming tho theek chha!
Bot: mi ek sahayak kumawati chhu


In [ ]:
while True:

    user = input("You: ")

    if user.lower() in ["exit", "quit"]:
        break

    chat(user)

You: hello?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: namaskar!
You: how is farming


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: farming tho theek chha
You: what is ur name


...

Bot: aaj mausam bhali chhao ✌️
You: how is hot


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: tho gaun chha!
You: who are you


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Bot: mi ek sahayak kumawati chhu
You: exit


In [ ]:
!zip -r kumaoni_llm_final.zip /content/drive/MyDrive/kumaoni_full_model